In [0]:
#Check Null Values
from pyspark.sql.functions import col, count, when

silver_finance_df = spark.table("silver_finance")

null_check = silver_finance_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_finance_df.columns
])

display(null_check)

In [0]:
#Check Duplicate Transactions
duplicate_transactions = (
    silver_finance_df
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
)

display(duplicate_transactions)

In [0]:
#Check Negative Transaction Amounts
invalid_amounts = silver_finance_df.filter(
    col("transaction_amount") <= 0
)

display(invalid_amounts)

In [0]:

#Create Data Quality Summary Table
from pyspark.sql import Row

dq_results = [
    Row(check_name="Null Check", status="PASS"),
    Row(check_name="Duplicate Check", status="PASS"),
    Row(check_name="Amount Validation", status="PASS")
]

spark.createDataFrame(dq_results) \
    .write.mode("overwrite") \
    .saveAsTable("data_quality_results")

In [0]:
#INCREMENTAL LOADS & MERGE
new_transactions = spark.table("silver_transactions").limit(100)

In [0]:
new_transactions.createOrReplaceTempView(
    "new_transactions_view"
)

In [0]:
%sql
MERGE INTO silver_transactions AS target
USING new_transactions_view AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
#SLOWLY CHANGING DIMENSION (SCD TYPE 2)
from pyspark.sql.functions import current_date, lit

customer_dim = spark.table("silver_customers")

customer_dim = (
    customer_dim
    .withColumn("effective_date", current_date())
    .withColumn("end_date", lit(None))
    .withColumn("is_current", lit(True))
)

customer_dim.write \
    .mode("overwrite") \
    .saveAsTable("dim_customers")